Bei anderen NB doppelte resample ünberprüfen dank prune

## Imports

In [59]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



## Load Data

In [60]:
max_hist = 13
is_local_data = False
Month_idx = 4
safe = True
start_wanted = None  # later this will be shifted, if it is to close to the beginning of the data, such that there is allways data also for the hist dimension
end = None
max_depth = 3


In [61]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [62]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [63]:

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
approx_maximas = max_v


In [64]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

 

In [ ]:
#min_start 

data_start = pd.to_datetime(raw_mrsol_for_mean.time[0].item())
offset = pd.tseries.frequencies.to_offset("ME")
min_start = data_start + pd.DateOffset(months=max_hist)

if start_wanted is None:
    start = min_start
else:
    start = pd.to_datetime(start_wanted) 
    start = max(min_start,start)
start = start.strftime("%Y-%m-%d")  

some explantions
slice 0-4 weil im moment die letzte schicht hartnäckig probleme macht

In [66]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas).clip(max = 1-1e-15)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [67]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [68]:
# Transform already as one function
#model.transform.Logit_Transform_ds()

In [69]:
def shape_input(ds, chunk_mask, hist):
    ds = model.shape_data.add_hist_dimension(ds, hist)    
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [70]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_2).sum(),#(chunk_mask != chunk_mask_1).sum(),
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [71]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [72]:
mean_predictor_sets  = {}
mean_max_predictor = shape_input(raw_input_for_mean,chunk_mask,max_hist)

for hist in range(1, max_hist):
    mean_predictor_sets[f"hist_{hist}"] = mean_max_predictor.isel(hist=slice(0,hist))

In [73]:
mean_predictor_sets["hist_1"]

<xarray.Dataset> Size: 8MB
Dimensions:  (hist: 1, time: 164, gridcell: 2935)
Coordinates:
  * hist     (hist) int64 8B 0
  * time     (time) datetime64[ns] 1kB 1851-04-30 1852-04-30 ... 2014-04-30
    lat      (gridcell) float64 23kB -56.25 -56.25 -56.25 ... 81.25 81.25 81.25
    lon      (gridcell) float64 23kB 288.8 291.2 293.8 ... 293.8 296.2 298.8
    height   float64 8B 2.0
Dimensions without coordinates: gridcell
Data variables:
    tas      (time, hist, gridcell) float64 4MB 278.4 277.9 ... 246.0 246.9
    pr       (time, hist, gridcell) float64 4MB 4.301e-05 ... 2.958e-06
Attributes: (12/52)
    CDI:                       Climate Data Interface version 1.9.6 (http://m...
    history:                   Thu Dec 19 16:54:57 2019: cdo -O -b F64 -remap...
    source:                    MPI-ESM1.2-LR (2017): \naerosol: none, prescri...
    institution:               Max Planck Institute for Meteorology
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    ...                        ...
    cmor_version:              3.5.0
    tracking_id:               hdl:21.14100/6b679cba-17b8-45eb-90dc-23d170c1998c
    cmip6-ng:                  \ncontact = cmip6-archive@env.ethz.ch\ndescrip...
    original_file_names:       /net/atmos/data/cmip6/historical/Amon/tas/MPI-...
    original_file_hash_codes:  44b9ee9e68daceb1f50e9680dcfb7744f0e272d73c5e18...
    CDO:                       Climate Data Operators version 1.9.6 (http://m...

In [74]:
Regr_set_mean = {}

In [75]:
#mean_target_da

In [76]:
for key,predictors in mean_predictor_sets.items():
    Regr_set_mean[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist"])
    Regr_set_mean[key].fit(predictors=predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

In [77]:
#Regr_set_mean["kontrolle"] = model.stats._parallel_linear_regression.ParLinearRegression()
#Regr_set_mean["kontrolle"].fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

Hier wäre die meinung einen neuen Run zu verwenden, die frage ist ob sich die variance durch das fehlen des runs zu tief ausfällt. Overfitting korrektur mit 1/(1-param/n_samples)^2? (SPäter probieren)

In [78]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)#Hier können noch sehr grosse werte auftauchen, wenn in irgendwelchen schichten die Maximas der verschieden runs sehr unterschiedlich sind.

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [79]:
var_predictor_sets  = {}
var_max_predictor = shape_input(raw_input_for_var,chunk_mask,max_hist)

for hist in range(1, max_hist):
    var_predictor_sets[f"hist_{hist}"] = var_max_predictor.isel(hist=slice(0,hist))

In [80]:
Regr_set_mean["hist_1"].params.pr

<xarray.DataArray 'pr' (gridcell: 2935, depth: 3, hist: 1)> Size: 70kB
array([[[ 7.98588600e+03],
        [ 3.34830859e+02],
        [ 4.45836385e+01]],

       [[ 1.45597061e+04],
        [ 1.49635270e+03],
        [-1.60171542e+00]],

       [[ 1.86633193e+04],
        [ 2.70559902e+03],
        [-8.60997163e-25]],

       ...,

       [[ 3.61680586e+01],
        [ 8.18888268e+01],
        [ 6.80896864e+02]],

       [[ 4.94930666e+02],
        [ 7.02874692e+02],
        [ 2.44358183e+03]],

       [[ 1.01098882e+02],
        [ 2.42854741e+02],
        [ 1.53923551e+03]]], shape=(2935, 3, 1))
Coordinates:
  * gridcell  (gridcell) int64 23kB 0 1 2 3 4 5 ... 2930 2931 2932 2933 2934
  * depth     (depth) float64 24B 0.03 0.19 0.78
  * hist      (hist) int64 8B 0

In [81]:
var_predictor_sets["hist_1"].pr 

<xarray.DataArray 'pr' (time: 164, hist: 1, gridcell: 2935)> Size: 4MB
array([[[5.28679637e-05, 4.70368287e-05, 4.16524441e-05, ...,
         2.31171899e-06, 2.03635889e-06, 1.91580487e-06]],

       [[6.12184949e-05, 4.29392715e-05, 3.46196323e-05, ...,
         2.23431613e-06, 2.20356349e-06, 2.04957445e-06]],

       [[6.36913563e-05, 4.57789773e-05, 3.19547862e-05, ...,
         5.08014001e-06, 5.82106351e-06, 6.83842006e-06]],

       ...,

       [[5.51163468e-05, 5.00483550e-05, 4.50342706e-05, ...,
         3.34773165e-06, 3.77216050e-06, 3.72080556e-06]],

       [[4.35360304e-05, 3.69620000e-05, 3.54597339e-05, ...,
         1.11529721e-06, 1.11319676e-06, 1.04716085e-06]],

       [[5.95045147e-05, 4.39408146e-05, 3.29656634e-05, ...,
         2.12069945e-06, 2.04910389e-06, 1.82411783e-06]]],
      shape=(164, 1, 2935))
Coordinates:
  * time     (time) datetime64[ns] 1kB 1851-04-30 1852-04-30 ... 2014-04-30
  * hist     (hist) int64 8B 0
    lat      (gridcell) float64 23kB -56.25 -56.25 -56.25 ... 81.25 81.25 81.25
    lon      (gridcell) float64 23kB 288.8 291.2 293.8 ... 293.8 296.2 298.8
    height   float64 8B 2.0
Dimensions without coordinates: gridcell
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [82]:
Regr_set_mean["hist_1"].params.pr * var_predictor_sets["hist_1"].pr 

<xarray.DataArray 'pr' (gridcell: 2935, depth: 3, hist: 1, time: 164)> Size: 12MB
array([[[[ 4.22197531e-01,  4.88883922e-01,  5.08631911e-01, ...,
           4.40152862e-01,  3.47673776e-01,  4.75196271e-01]],

        [[ 1.77018257e-02,  2.04978412e-02,  2.13258315e-02, ...,
           1.84546537e-02,  1.45772065e-02,  1.99239478e-02]],

        [[ 2.35704618e-03,  2.72934325e-03,  2.83959240e-03, ...,
           2.45728728e-03,  1.94099464e-03,  2.65292777e-03]]],


       [[[ 6.84842403e-01,  6.25183174e-01,  6.66528456e-01, ...,
           7.28689340e-01,  5.38155857e-01,  6.39765347e-01]],

        [[ 7.03836856e-02,  6.42522947e-02,  6.85014963e-02, ...,
           7.48899910e-02,  5.53081884e-02,  6.57509565e-02]],

        [[-7.53396140e-05, -6.87764933e-05, -7.33248940e-05, ...,
          -8.01632220e-05, -5.92026054e-05, -7.03806803e-05]]],


...

       [[[ 1.00785646e-03,  1.09061114e-03,  2.88102284e-03, ...,
           1.86695791e-03,  5.50955214e-04,  1.01416435e-03]],

        [[ 1.43130513e-03,  1.54882901e-03,  4.09147822e-03, ...,
           2.65135615e-03,  7.82437829e-04,  1.44026326e-03]],

        [[ 4.97600958e-03,  5.38458770e-03,  1.42242450e-02, ...,
           9.21758286e-03,  2.72018738e-03,  5.00715303e-03]]],


       [[[ 1.93685730e-04,  2.07209685e-04,  6.91356620e-04, ...,
           3.76169281e-04,  1.05866790e-04,  1.84416273e-04]],

        [[ 4.65262295e-04,  4.97748871e-04,  1.66074273e-03, ...,
           9.03615271e-04,  2.54307976e-04,  4.42995664e-04]],

        [[ 2.94887488e-03,  3.15477776e-03,  1.05259390e-02, ...,
           5.72719603e-03,  1.61182716e-03,  2.80774694e-03]]]],
      shape=(2935, 3, 1, 164))
Coordinates:
  * gridcell  (gridcell) int64 23kB 0 1 2 3 4 5 ... 2930 2931 2932 2933 2934
    lat       (gridcell) float64 23kB -56.25 -56.25 -56.25 ... 81.25 81.25 81.25
    lon       (gridcell) float64 23kB 288.8 291.2 293.8 ... 293.8 296.2 298.8
  * depth     (depth) float64 24B 0.03 0.19 0.78
  * hist      (hist) int64 8B 0
  * time      (time) datetime64[ns] 1kB 1851-04-30 1852-04-30 ... 2014-04-30
    height    float64 8B 2.0
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [83]:
residuals = {}
for key, regr in Regr_set_mean.items():
    residuals[key] = regr.residuals(var_predictor_sets[key], var_target,location_dim="gridcell", regr_dim="time")


### Linear Regression of the Variance

In [84]:
Regr_set_var = {}

In [85]:
for key, res in residuals.items():
    Regr_set_var[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist"])
    Regr_set_var[key].fit(predictors=var_predictor_sets[key], target=(res.residuals)**2,location_dim="gridcell", regr_dim="time")

for simplicity not in use
### Compute skewness samples
skew_target_noneT = shape_target(raw_mrsol_for_skew, approx_maximas)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Logit_Transform_ds(skew_target_noneT)

skew_predictors = shape_input(raw_input_for_skew,chunk_mask)
mean_prediction = LinReg_mean.predict(skew_predictors)
residuals = skew_target.mrsol - mean_prediction.prediction
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))
standardized_values_for_skew = (residuals/sigmas)
### Linear Regression of the Skewness
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Predictor

### Export Prameters

In [86]:
if safe:
    for key, mean_regr in Regr_set_mean.items():
        model.save.save_params(mean_regr.params,Regr_set_var[key].params, maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/history/{key}/local{is_local_data}/month{Month_idx}", name=f"start={start_wanted},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
